# Data Augmentation v2 — Sinh Thêm Câu Hỏi Đa Dạng

## Mục tiêu
- Round 1 (đã xong): `n_questions=4` × 1,861 chunks → ~6,600 pairs  
- **Round 2 (notebook này):** sinh thêm 4 câu hỏi **khác phong cách** cho mỗi chunk  
- Kết quả: ~12,000-13,000 pairs → fine-tune bi-encoder v4.2

## Điểm khác biệt so với round 1
- Prompt yêu cầu **phong cách khác**: tình huống cụ thể, câu hỏi có/không, câu hỏi so sánh
- Skip chunks đã có ≥ 6 queries (đã đủ đa dạng)
- Output: `train_aug2.jsonl` → merge với `train.jsonl` gốc

In [ ]:
# CELL 1 - Install
!python -m pip -q install -U "transformers>=4.41.0" "accelerate>=0.30.0" "bitsandbytes>=0.46.1" "sentencepiece"

In [ ]:
# CELL 2 - Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CELL 3 — Config paths
import os

BASE_DIR    = "/content/drive/MyDrive/Legal chat bot/Data"
CHUNKS_PATH = os.path.join(BASE_DIR, "chunks.json")
TRAIN_PATH  = os.path.join(BASE_DIR, "train.jsonl")       # round 1 output
OUT_AUG2    = os.path.join(BASE_DIR, "train_aug2.jsonl")  # round 2 output
OUT_MERGED  = os.path.join(BASE_DIR, "train_merged.jsonl")# merged output
OUT_MERGED_NEG = os.path.join(BASE_DIR, "train_merged_with_neg.jsonl")

CHECKPOINT_FILE = os.path.join(BASE_DIR, "aug2_checkpoint.txt")

N_EXTRA     = 4    # thêm 4 câu hỏi/chunk
SKIP_THRESH = 6    # bỏ qua chunk đã có >= 6 queries (đủ đa dạng rồi)
CKPT_EVERY  = 25   # checkpoint mỗi 25 chunks

print(f"Config: +{N_EXTRA} queries/chunk, skip if >= {SKIP_THRESH} existing")

In [ ]:
# CELL 4 — Load chunks + đếm queries hiện tại mỗi chunk
import json

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)
print(f"Total chunks: {len(chunks)}")

# Đếm queries hiện tại theo chunk_index
query_count = {}  # chunk_index → count
existing_pairs = []
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line.strip())
        ci = (row.get("meta") or {}).get("chunk_index")
        if ci is not None:
            query_count[ci] = query_count.get(ci, 0) + 1
        existing_pairs.append(row)

print(f"Existing train pairs: {len(existing_pairs)}")
print(f"Chunks with queries: {len(query_count)}")
skip_count = sum(1 for v in query_count.values() if v >= SKIP_THRESH)
to_aug = sum(1 for v in query_count.values() if v < SKIP_THRESH)
no_query = len(chunks) - len(query_count)
print(f"  Skip (>= {SKIP_THRESH} queries): {skip_count}")
print(f"  Augment (< {SKIP_THRESH} queries): {to_aug}")
print(f"  No queries yet: {no_query}")
print(f"Estimated new pairs: ~{(to_aug + no_query) * N_EXTRA}")

In [ ]:
# CELL 5 — Load Qwen2.5-7B-Instruct (same as round 1)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
model.config.use_cache = False
print("Model loaded. Device:", next(model.parameters()).device)

In [ ]:
# CELL 6 — Prompt v2: KHÁC PHONG CÁCH so với round 1
import json, re

SYSTEM_MSG = (
    "Bạn là công cụ sinh câu hỏi CHỈ BẰNG TIẾNG VIỆT dựa trên đoạn trích pháp luật Việt Nam. "
    "BẮT BUỘC: chỉ dùng tiếng Việt (có dấu), TUYỆT ĐỐI KHÔNG dùng tiếng Trung/English hay ký tự Hán. "
    "Không giải thích. Không markdown. Không thêm text ngoài JSON."
)

def build_prompt_v2(chunk, n_questions=4):
    """Prompt khác phong cách: tình huống, có/không, so sánh, điều kiện."""
    md = chunk.get("metadata", {}) or {}
    doc = md.get("van_ban", "Văn bản pháp luật")
    dieu = md.get("dieu")
    khoan = md.get("khoan")
    ref_parts = []
    if dieu:   ref_parts.append(f"Điều {dieu}")
    if khoan:  ref_parts.append(f"Khoản {khoan}")
    ref = " - ".join(ref_parts) if ref_parts else "Không rõ tham chiếu"
    passage = chunk.get("text", "")

    return f"""Văn bản: {doc}
Tham chiếu: {ref}

Đoạn trích:
\"\"\"{passage}\"\"\"

Yêu cầu:
- Sinh {n_questions} câu hỏi tiếng Việt TỰ NHIÊN và ĐA DẠNG:
  * 1-2 câu hỏi TÌNH HUỐNG (VD: "Nếu... thì...", "Khi... thì...")
  * 1-2 câu hỏi CÓ/KHÔNG (VD: "Có phải... không?", "... có được phép... không?")
  * 0-1 câu hỏi SO SÁNH hoặc ĐIỀU KIỆN (VD: "Điều kiện để...", "Khác nhau giữa...")
- Câu hỏi phải trả lời được chỉ dựa trên đoạn trích.
- Không nhắc "đoạn trích", "đoạn văn", "theo quy định trên".
- TUYỆT ĐỐI KHÔNG dùng tiếng Trung/English hoặc ký tự Hán.

Trả về DUY NHẤT JSON đúng schema:
{{"queries": ["...", "..."]}}""".strip()


HAN_RE = re.compile(r"[\u3400-\u4DBF\u4E00-\u9FFF]")
BAD_PATTERNS = [
    r"\bbạn là\b", r"\bdựa vào\b", r"\bđoạn trích\b",
    r"\bđoạn văn\b", r"\btheo quy định trên\b", r"\bsinh câu hỏi\b"
]

def is_valid_query(q: str) -> bool:
    q = q.strip()
    if len(q) < 10: return False
    if HAN_RE.search(q): return False
    low = q.lower()
    for pat in BAD_PATTERNS:
        if re.search(pat, low): return False
    return True

def safe_parse(text):
    s = text.strip()
    try:
        data = json.loads(s)
        qs = data.get("queries", [])
        if isinstance(qs, list):
            return [q.strip() for q in qs if isinstance(q, str) and len(q.strip()) >= 10]
    except: pass
    last_r = text.rfind("}")
    if last_r == -1: return []
    for start in range(text.rfind("{", 0, last_r), -1, -1):
        if text[start] != "{": continue
        blob = text[start:last_r+1]
        try:
            data = json.loads(blob)
            qs = data.get("queries", [])
            if isinstance(qs, list):
                return [q.strip() for q in qs if isinstance(q, str) and len(q.strip()) >= 10]
        except: continue
    return []

print("Prompt v2 + utils ✓")

In [ ]:
# CELL 7 — Generate function (same logic as round 1)
import torch

@torch.no_grad()
def generate_queries_v2(chunk, n_questions=4, max_new_tokens=300,
                        temperature=0.3, max_retries=2):
    user_prompt = build_prompt_v2(chunk, n_questions=n_questions)

    def _gen_once(sys_msg, temp):
        messages = [
            {"role": "system", "content": sys_msg},
            {"role": "user", "content": user_prompt},
        ]
        enc = tokenizer.apply_chat_template(
            messages, tokenize=True,
            add_generation_prompt=True, return_tensors="pt"
        )
        if isinstance(enc, torch.Tensor):
            input_ids = enc.to(model.device)
            attention_mask = None
            prompt_len = input_ids.shape[-1]
        else:
            input_ids = enc["input_ids"].to(model.device)
            attention_mask = enc.get("attention_mask")
            if attention_mask is not None:
                attention_mask = attention_mask.to(model.device)
            prompt_len = input_ids.shape[-1]

        out = model.generate(
            input_ids=input_ids, attention_mask=attention_mask,
            max_new_tokens=max_new_tokens, do_sample=True,
            temperature=temp, top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )
        gen_ids = out[0][prompt_len:]
        return tokenizer.decode(gen_ids, skip_special_tokens=True)

    out_text = _gen_once(SYSTEM_MSG, temperature)
    tries = 0
    while tries < max_retries and HAN_RE.search(out_text or ""):
        tries += 1
        strict_msg = SYSTEM_MSG + " VI PHẠM. Xuất lại CHỈ tiếng Việt, KHÔNG ký tự Hán. JSON only."
        out_text = _gen_once(strict_msg, temp=max(0.05, temperature - 0.1 * tries))
    return out_text

print("generate_queries_v2 ✓")

In [ ]:
# CELL 8 — Main loop: generate thêm queries (resume-capable)
import time, gc

# Load checkpoint
start_i = 0
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        start_i = int(f.read().strip())
    print(f"Resume from chunk i = {start_i + 1} (1-based)")
else:
    print("Starting fresh")

# Đếm output hiện tại
aug2_count = 0
if os.path.exists(OUT_AUG2):
    with open(OUT_AUG2) as f:
        aug2_count = sum(1 for _ in f)
print(f"Aug2 lines so far: {aug2_count}")

t_global = time.time()

with open(OUT_AUG2, "a", encoding="utf-8") as fout:
    for i, chunk in enumerate(chunks):
        if i < start_i:
            continue  # skip đã xử lý

        ci = i  # chunk_index
        existing = query_count.get(ci, 0)

        # Skip nếu đã có đủ queries
        if existing >= SKIP_THRESH:
            continue

        passage = chunk.get("text", "").strip()
        if not passage:
            continue

        meta = chunk.get("metadata", {}) or {}

        print(f"[{i+1}/{len(chunks)}] generating (has {existing} queries)...", flush=True)

        try:
            raw = generate_queries_v2(chunk, n_questions=N_EXTRA)
            queries = safe_parse(raw)
            queries = [q for q in queries if is_valid_query(q)]
        except Exception as e:
            print(f"  ERROR: {e}")
            queries = []

        for q in queries:
            row = {
                "query": q, "passage": passage, "label": 1,
                "meta": {
                    "van_ban": meta.get("van_ban", ""),
                    "chuong": meta.get("chuong"),
                    "dieu": meta.get("dieu"),
                    "khoan": meta.get("khoan"),
                    "diem": meta.get("diem"),
                    "chunk_index": ci,
                    "source": "aug2"
                }
            }
            fout.write(json.dumps(row, ensure_ascii=False) + "\n")
        fout.flush()

        # Checkpoint
        if (i + 1) % CKPT_EVERY == 0:
            elapsed = round(time.time() - t_global, 1)
            print(f"Checkpoint at chunk {i+1}. Elapsed: {elapsed}s", flush=True)
            with open(CHECKPOINT_FILE, "w") as f:
                f.write(str(i))

        gc.collect()

    # Final checkpoint
    with open(CHECKPOINT_FILE, "w") as f:
        f.write(str(len(chunks)))

print(f"\nDone! Wrote: {OUT_AUG2}")

In [ ]:
# CELL 9 — Merge train.jsonl + train_aug2.jsonl → train_merged.jsonl
import json

seen_pairs = set()  # (query, passage) để dedup
total_written = 0
dup_count = 0

with open(OUT_MERGED, "w", encoding="utf-8") as fout:
    # Round 1 trước
    for src in [TRAIN_PATH, OUT_AUG2]:
        source_name = "round1" if src == TRAIN_PATH else "aug2"
        src_count = 0
        with open(src, "r", encoding="utf-8") as fin:
            for line in fin:
                line = line.strip()
                if not line: continue
                try:
                    row = json.loads(line)
                except:
                    continue
                q = row.get("query", "").strip()
                p = row.get("passage", "").strip()
                key = (q[:100], p[:100])  # dedup bằng 100 ký tự đầu
                if key in seen_pairs:
                    dup_count += 1
                    continue
                seen_pairs.add(key)
                fout.write(json.dumps(row, ensure_ascii=False) + "\n")
                total_written += 1
                src_count += 1
        print(f"  {source_name}: {src_count} unique pairs")

print(f"\nMerged → {OUT_MERGED}")
print(f"Total unique pairs: {total_written}")
print(f"Duplicates removed: {dup_count}")
print(f"Expected gain: +{total_written - len(existing_pairs)} new pairs")

In [ ]:
# CELL 10 — Tạo train_merged_with_neg.jsonl (same logic as original Cell 12)
import json, random, os

random.seed(42)

# Build index chunks theo van_ban
by_doc = {}
for idx, ch in enumerate(chunks):
    doc = (ch.get("metadata") or {}).get("van_ban", "UNKNOWN")
    by_doc.setdefault(doc, []).append(idx)

def pick_negative(chunk_idx):
    ch = chunks[chunk_idx]
    doc = (ch.get("metadata") or {}).get("van_ban", "UNKNOWN")
    candidates = by_doc.get(doc, [])
    if len(candidates) >= 2:
        j = random.choice([x for x in candidates if x != chunk_idx])
    else:
        j = random.randrange(len(chunks))
        while j == chunk_idx:
            j = random.randrange(len(chunks))
    return chunks[j].get("text", "")

written = 0
with open(OUT_MERGED, "r", encoding="utf-8") as fin, \
     open(OUT_MERGED_NEG, "w", encoding="utf-8") as fout:
    for line in fin:
        line = line.strip()
        if not line: continue
        pos = json.loads(line)
        if pos.get("label", 1) == 0:
            continue  # bỏ negative cũ nếu có
        chunk_idx = (pos.get("meta") or {}).get("chunk_index")
        if chunk_idx is None:
            continue

        fout.write(json.dumps(pos, ensure_ascii=False) + "\n")

        neg = dict(pos)
        neg["passage"] = pick_negative(chunk_idx)
        neg["label"] = 0
        fout.write(json.dumps(neg, ensure_ascii=False) + "\n")
        written += 2

print(f"Wrote: {OUT_MERGED_NEG}")
print(f"Total lines (pos + neg): {written}")

In [ ]:
# CELL 11 — Thống kê cuối
import json

def count_lines(path):
    if not os.path.exists(path): return 0
    with open(path) as f: return sum(1 for _ in f)

r1  = count_lines(TRAIN_PATH)
r2  = count_lines(OUT_AUG2)
mg  = count_lines(OUT_MERGED)
mg_neg = count_lines(OUT_MERGED_NEG)

print("="*55)
print(f"  {'File':<35} {'Pairs':>10}")
print("="*55)
print(f"  train.jsonl (round 1)            {r1:>10,}")
print(f"  train_aug2.jsonl (round 2)       {r2:>10,}")
print(f"  train_merged.jsonl (unique)      {mg:>10,}  ← dùng để train")
print(f"  train_merged_with_neg.jsonl      {mg_neg:>10,}  ← dùng cho CE")
print("="*55)
print(f"\n  Tăng training data: {r1} → {mg} pairs (+{mg-r1})")
print(f"  Tỷ lệ tăng: {(mg/r1-1)*100:.1f}%")
print(f"\n  → Sao chép train_merged.jsonl vào thư mục cross-encoder/data/")
print(f"    đổi tên thành train.jsonl rồi chạy baseline_v4_finetune_bi.ipynb")